# 🎓 Phase 6 · Lesson 00: Welcome, Setup & Your First Ensemble

**Course:** Ensemble Learning & Model Optimization (Weeks 11–12)  
**Time:** ~60–90 minutes  **Prerequisites:** Python, pandas, train/test split, linear regression, decision trees

---

### 👋 Welcome!

Hello, and welcome to Phase 6. I'm going to be your instructor for the next two weeks, and I want to start with a
confession: **the most powerful idea in practical machine learning is almost embarrassingly simple.**

> *A committee of average models can beat one brilliant model.*

That's it. That's the whole course. Everything else (bagging, boosting, stacking, XGBoost, SHAP) is
*how* to build the committee, *how* to tune it, and *how* to explain what it decided.

If you've ever watched a quiz-show audience vote on the answer to a tricky question and beat the smartest contestant,
you already understand ensembles. By the end of these two weeks you'll be able to **build** that audience,
**tune** it, **compare** it fairly against alternatives, and **defend** your choice with evidence.

### 🎯 What you will be able to do after this lesson
1. Check that your environment is ready for the whole course.
2. Take a guided tour of the three datasets we'll use.
3. Refresh the evaluation metrics (RMSE, MAE, R²) *by hand*.
4. Run your very first ensemble and see (with your own eyes) why we bother.

## 🗺️ The map of the next two weeks

| Lesson | Topic | Big question |
|---|---|---|
| **00** | Welcome & setup | Is everything installed? What does the data look like? |
| **01** | Why ensembles work | *Why* does averaging many models help? (bias, variance, correlation) |
| **02** | Bagging & Random Forest | How do we build many *different* models from one dataset? |
| **03** | Boosting & Gradient Boosting | What if each model learns from the previous one's mistakes? |
| **04** | XGBoost · LightGBM · CatBoost | What makes the modern libraries so fast and so accurate? |
| **05** | Voting & Stacking | Can we combine *different kinds* of models? |
| **06** | Cross-validation & model comparison | How do we know model A is *really* better than model B? |
| **07** | Hyperparameter tuning | Grid, random, Bayesian: how do we search smartly? |
| **08** | Interpretability | *Why* did the model predict that? (importance, permutation, SHAP) |
| **09** | **Practical Assignment** | Five regression models, one dataset, evidence-based winner |
| **10** | **Capstone Project** | End-to-end project on a fresh dataset |

### 📜 How to take this course (my three rules)

1. **Type the code. Don't just run it.** Your fingers learn things your eyes skip.
2. **Predict before you run.** Whenever you see 🔮, say your prediction out loud (or write it down) *first*. Being wrong is where learning lives.
3. **Break things on purpose.** Change a number. Delete a line. Ask "what if?". Notebooks are free; curiosity is the point.

### 🔣 Icons you'll see

| icon | meaning |
|---|---|
| 🎯 | learning goal |
| 💡 | an intuition or analogy |
| 🔮 | **predict first**, then run |
| ⚠️ | a classic mistake: read twice |
| 🧪 | worked example (I solve it, you watch) |
| 🧩 | problem (you solve it; hints are hidden in dropdowns) |
| ✋ | quick concept check |

## 1 · Is your environment ready?

Let's make sure all the libraries are installed. If something says `NOT INSTALLED`, open a terminal in the course
folder and run:

```bash
pip install -r requirements.txt
```

(Then restart the notebook kernel.) We'll set up some course helpers first: the first cell of **every** notebook looks like this.

In [1]:
import sys; sys.path.append("../utils")          # lets us import the course helpers
from course_utils import *                       # load_housing(), check(), set_style(), ...
import numpy as np, pandas as pd, matplotlib.pyplot as plt

set_style()
np.random.seed(SEED)

In [2]:
environment_report()

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,library,version
0,python,3.12.3
1,numpy,2.5.4
2,pandas,3.0.6
3,scipy,1.18.1
4,sklearn,1.9.1
5,matplotlib,3.11.2
6,xgboost,3.4.1
7,lightgbm,4.7.0
8,catboost,1.2.10
9,shap,0.54.1


> ⚠️ **Version note.** The notebooks were written and tested against recent releases (see the table above).
> If you have older versions and see strange errors, upgrade with `pip install -U scikit-learn xgboost lightgbm catboost shap optuna`.

## 2 · Meet the data

We use three **synthetic** datasets (generated by `utils/make_datasets.py`). Synthetic data has a superpower:
*we know the recipe that generated it*, so later we can check whether a model discovered the true drivers of the target.
They're also deliberately messy (missing values, outliers, skew, non-linear effects) just like the real world.

| dataset | task | target | used in |
|---|---|---|---|
| `housing.csv` | regression | `price_k` (house price, $ thousands) | Lessons 01–09 (main dataset) |
| `loan_default.csv` | classification | `defaulted` (0/1, ~14% positive) | Lessons 05, 06, 08 |
| `crop_yield.csv` | regression | `yield_t_ha` (tonnes per hectare) | **Capstone project** |

Let's meet **housing**, our workhorse.

In [3]:
housing = load_housing()
print("shape:", housing.shape)
housing.head()

shape: (3000, 19)


,listing_id,year_listed,neighborhood,property_type,size_sqm,bedrooms,bathrooms,age_years,renovated,floor,distance_to_center_km,transit_minutes,school_rating,security_rating,has_parking,has_garden,has_pool,energy_rating,price_k
0,10001,2023,Outer Meadows,detached,289.0,7,5,2.0,0,0,23.60,33.0,4.8,4.0,1,1,0,F,462.4
1,10002,2019,Market District,apartment,92.0,2,2,0.0,1,5,1.42,1.0,5.6,3.0,0,0,0,B,356.9
2,10003,2023,Industrial East,apartment,60.0,2,1,25.0,0,4,0.96,7.0,5.0,3.0,0,0,0,C,128.8
3,10004,2022,Parkside,detached,163.0,4,3,5.0,0,0,1.94,2.0,10.0,4.0,1,1,0,F,976.9
4,10005,2019,Riverside,apartment,108.0,3,1,0.0,0,15,6.56,4.0,8.2,5.0,1,0,0,C,393.7


In [4]:
# Column types and missing values -- always the first thing to check
summary = pd.DataFrame({
    "dtype": housing.dtypes.astype(str),
    "n_unique": housing.nunique(),
    "missing_%": (housing.isna().mean() * 100).round(1),
})
summary

,dtype,n_unique,missing_%
listing_id,int64,3000,0.0
year_listed,int64,6,0.0
neighborhood,str,12,0.0
property_type,str,3,0.0
size_sqm,float64,312,0.0
bedrooms,int64,7,0.0
bathrooms,int64,6,0.0
age_years,float64,80,3.9
renovated,int64,2,0.0
floor,int64,19,0.0


**Notice anything?** Four columns have missing values (`age_years`, `school_rating`, `security_rating`, `energy_rating`).
In real data, *how* values go missing often carries information. Here, school ratings are missing more often in lower-priced
neighbourhoods. Different models cope with this differently: **tree-based boosting libraries can handle NaNs natively**,
which is one of their practical superpowers (Lesson 04).

### The target: price

🔮 **Predict:** will the price histogram be symmetric (bell-shaped) or skewed? Which way?

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(housing["price_k"], bins=60, color=PALETTE["blue"], alpha=.85)
axes[0].set_title("price_k  (raw)"); axes[0].set_xlabel("price, $ thousands")
axes[1].hist(np.log(housing["price_k"]), bins=60, color=PALETTE["orange"], alpha=.85)
axes[1].set_title("log(price_k)"); axes[1].set_xlabel("log price")
plt.tight_layout(); plt.show()

print(f"skew of price: {housing['price_k'].skew():.2f}   |   skew of log(price): {np.log(housing['price_k']).skew():.2f}")

skew of price: 2.52   |   skew of log(price): 0.25


💡 **Right-skewed**, as nearly all prices are: a few mansions drag a long tail to the right. Taking the logarithm
turns *multiplicative* effects ("a pool adds 12%") into *additive* ones, and tames outliers. We'll come back to this
when we compare models in Lesson 09: it matters a lot for linear models and somewhat less for trees.

### Which features seem to matter?
Let's look at three quick relationships before any modelling.

In [6]:
import seaborn as sns
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(housing["size_sqm"], housing["price_k"], s=6, alpha=.35)
axes[0].set_xlabel("size_sqm"); axes[0].set_ylabel("price_k"); axes[0].set_title("Bigger → pricier (fanning out!)")

order = housing.groupby("neighborhood")["price_k"].median().sort_values().index
sns.boxplot(data=housing, y="neighborhood", x="price_k", order=order, showfliers=False, color=PALETTE["blue"], ax=axes[1])
axes[1].set_title("Price by neighbourhood"); axes[1].set_xlabel("price_k"); axes[1].set_ylabel("")

axes[2].scatter(housing["distance_to_center_km"], housing["price_k"], s=6, alpha=.35, color=PALETTE["green"])
axes[2].set_xlabel("distance to centre (km)"); axes[2].set_title("Farther → cheaper (but not linearly)")
plt.tight_layout(); plt.show()

💡 Two things to notice. First, the scatter in the left panel **fans out**: bigger houses have more *variable* prices.
(That's multiplicative structure, and a hint that predicting `log(price)` may help.) Second, none of these relationships
is a clean straight line. Hold that thought: **models that bend, and combinations of models, are going to shine.**

## 3 · Refresher: how do we *score* a regression model?

Before we build committees, let's make sure we can grade them. Suppose a model predicted prices for five houses:

In [7]:
y_true = np.array([300, 450, 250, 800, 500])     # actual prices ($k)
y_pred = np.array([320, 400, 270, 700, 510])     # model predictions

errors = y_pred - y_true
print("errors        :", errors)
print("abs errors    :", np.abs(errors))
print("squared errors:", errors ** 2)

errors        : [  20  -50   20 -100   10]
abs errors    : [ 20  50  20 100  10]
squared errors: [  400  2500   400 10000   100]


### 🧪 Worked Example 0.1: computing the metrics by hand

| metric | formula | in words |
|---|---|---|
| **MAE** | $\frac{1}{n}\sum_i \lvert e_i \rvert$ | the *average* size of a mistake |
| **RMSE** | $\sqrt{\frac{1}{n}\sum_i e_i^2}$ | like MAE but **punishes big mistakes more** (they get squared) |
| **R²** | $1 - \dfrac{\sum_i e_i^2}{\sum_i (y_i - \bar y)^2}$ | fraction of the variance explained, compared with "always predict the mean" |

**Step 1: MAE.** $(20+50+20+100+10)/5 = 40$.  
**Step 2: RMSE.** Squared errors: $400+2500+400+10000+100 = 13400$; divide by 5 → 2680; square root → ≈ **51.8**.  
**Step 3: R².** The mean of `y_true` is 460. Total variation $= 160^2+(-10)^2+210^2+340^2+40^2 = 25600+100+44100+115600+1600 = 187000$. So $R^2 = 1 - 13400/187000 \approx 0.928$.

Notice **RMSE (51.8) > MAE (40)**: that one $100k miss on the $800k house is punished more by RMSE. Whenever RMSE ≫ MAE, you have a few big errors hiding somewhere.
Let's confirm with code.

In [8]:
mae  = np.mean(np.abs(errors))
rmse = np.sqrt(np.mean(errors ** 2))
r2   = 1 - np.sum(errors ** 2) / np.sum((y_true - y_true.mean()) ** 2)
print(f"MAE  = {mae:.1f}\nRMSE = {rmse:.1f}\nR²   = {r2:.3f}")

# the one-liner from our helper library gives all four at once:
regression_report(y_true, y_pred)

MAE  = 40.0
RMSE = 51.8
R²   = 0.928


{'RMSE': 51.76871642217914,
 'MAE': 40.0,
 'R2': 0.9283422459893048,
 'MAPE_%': np.float64(8.055555555555555)}

⚠️ **Classic mistake #1: evaluating on the training data.** A model that memorises the training set scores
beautifully there and fails on new houses. We *always* hold out a **test set** the model never sees while learning.
We'll use one fixed split for the whole course (`get_housing_split()`: 80% train / 20% test, seed 42), so every result
you see is comparable to every other.

## 4 · 🚀 Your first ensemble (the 5-minute teaser)

Time for the payoff. We will compare:

* **Baseline**: always predict the average training price (the dumbest sensible model!)
* **One decision tree**: a single "expert"
* **Random Forest**: a committee of 300 trees (Lesson 02)
* **Gradient Boosting**: a *sequence* of trees, each correcting the last (Lesson 03)

🔮 **Predict before you run:** Rank these four from best to worst on test-set RMSE. How much better do you think the
forest is than the single tree: 5%? 20%? 50%?

In [9]:
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

X_train, X_test, y_train, y_test = get_housing_split()
print("train:", X_train.shape, " test:", X_test.shape)

models = {
    "Baseline (mean)":     DummyRegressor(strategy="mean"),
    "Single Decision Tree": DecisionTreeRegressor(random_state=SEED),
    "Random Forest":       RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1),
    "Gradient Boosting":   GradientBoostingRegressor(random_state=SEED),
}

rows = []
for name, model in models.items():
    pipe = make_pipeline(make_preprocessor(X_train), model)   # preprocessing + model in one object
    with Timer() as t:
        pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    rows.append({"model": name, **regression_report(y_test, pred), "fit_seconds": t.seconds})

results = results_table(rows)
results.round(3)

train: (2400, 17)  test: (600, 17)


,model,RMSE,MAE,R2,MAPE_%,fit_seconds
0,Gradient Boosting,108.249,64.830,0.908,15.049,0.418
1,Random Forest,122.808,78.631,0.881,18.806,3.817
2,Single Decision Tree,215.075,122.261,0.636,26.737,0.031
3,Baseline (mean),356.446,248.972,-0.000,83.946,0.024


In [10]:
fig, ax = plt.subplots(figsize=(8, 3.8))
colors = [PALETTE["gray"], PALETTE["red"], PALETTE["green"], PALETTE["blue"]]
order = results.sort_values("RMSE", ascending=False)
ax.barh(order["model"], order["RMSE"], color=[dict(zip(models, colors))[m] for m in order["model"]])
for i, v in enumerate(order["RMSE"]):
    ax.text(v + 3, i, f"{v:.0f}", va="center")
ax.set_xlabel("Test RMSE ($ thousands): lower is better"); ax.set_title("A committee beats a lone expert")
plt.tight_layout(); plt.show()

### 🔎 What just happened?

* The **baseline** tells us the floor: any model worth deploying must beat it (R² = 0 by definition).
* The **single tree** is much better than the baseline, but notice how it compares with the ensembles. A deep tree
  fits its training data almost perfectly (it has *high variance*): shown a slightly different sample, it would
  grow a noticeably different tree.
* The **forest** averages 300 such wobbly trees; the wobbles cancel out. The **boosted** model takes a different route
: it builds small trees *sequentially*, each fixing what the previous ones got wrong.

You've just witnessed the central empirical fact of this course: **ensembles of trees dominate tabular data.**
Over the next lessons we'll understand *why*, build them from scratch, and learn to squeeze out every last drop of performance
without fooling ourselves.

## 5 · Starter glossary

Skim now, revisit often. (A fuller one lives in `GLOSSARY.md`.)

| term | one-line meaning |
|---|---|
| **Estimator / model** | an object that learns from data (`.fit`) and predicts (`.predict`) |
| **Base learner / weak learner** | one member of the ensemble (usually a small decision tree) |
| **Bias** | systematic error: the model is too simple to capture the truth |
| **Variance** | sensitivity to the particular training sample: the model is too wiggly |
| **Overfitting** | great on training data, poor on new data |
| **Ensemble** | a combination of several models into one predictor |
| **Hyperparameter** | a setting *you* choose before training (tree depth, learning rate…) |
| **Cross-validation** | repeated train/validate splits for a more reliable score |
| **Data leakage** | information from the test set sneaking into training: the silent killer |

### ✋ Check your understanding

Try to answer *out loud* before you open the answer. If you can't say it simply, you don't own it yet.

**Q1. Why do we report RMSE *and* MAE instead of just one?**

<details><summary>Show answer</summary>

RMSE squares errors, so it is dominated by large mistakes; MAE treats all mistakes equally. If RMSE is much larger than MAE, a few big errors are hiding in your predictions. Looking at both tells you about the *shape* of your errors.

</details>

**Q2. A model gets R² = 0.95 on the training set and 0.62 on the test set. What's the diagnosis?**

<details><summary>Show answer</summary>

**Overfitting** (high variance): it memorised the training data. A big train–test gap is the signature. Ensembles like bagging are designed to attack exactly this.

</details>

**Q3. Why is a `DummyRegressor` baseline worth running?**

<details><summary>Show answer</summary>

It sets the floor. If a fancy model barely beats 'always predict the mean', either the features carry little signal or something is wrong with the pipeline. It also puts the other scores in perspective.

</details>

## 6 · Problems

Three warm-ups. Each has hidden hints: try for at least five minutes before opening them.
Run the cell under each problem: it grades you instantly.

### 🧩 Problem 00.1 ⭐: Missing data audit

What fraction of the **`school_rating`** values in `housing` are missing? Store it as a number between 0 and 1 in the variable `frac_missing`.

<details><summary>💡 Hint 1</summary>

Every pandas Series has `.isna()`, which gives True/False per row.

</details>

<details><summary>💡 Hint 2</summary>

The mean of a True/False column is the fraction of Trues: `housing['school_rating'].isna().mean()`.

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [ ]:
frac_missing = TODO   # YOUR CODE HERE

# ---- self-check (don't edit) ----
check("00.1", lambda: 0.04 < frac_missing < 0.08, "It should be around 0.06 (about 6%).", needs=[frac_missing])

### 🧩 Problem 00.2 ⭐: Score it by hand

A model predicts `[10, 20, 30, 40]` for houses whose true prices are `[12, 18, 33, 45]` (all $ thousands).  
Compute the **MAE** and the **RMSE** with NumPy (no sklearn!) and store them in `my_mae` and `my_rmse`.

<details><summary>💡 Hint 1</summary>

Errors are `pred - true`. For MAE take absolute values then the mean.

</details>

<details><summary>💡 Hint 2</summary>

For RMSE: square the errors, take the mean, then `np.sqrt`.

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [ ]:
true = np.array([12, 18, 33, 45])
pred = np.array([10, 20, 30, 40])

my_mae  = TODO   # YOUR CODE HERE
my_rmse = TODO   # YOUR CODE HERE

# ---- self-check (don't edit) ----
check("00.2", lambda: abs(my_mae - 3.0) < 1e-9 and abs(my_rmse - np.sqrt(10.5)) < 1e-9,
      "Errors are [-2, 2, -3, -5]. MAE = 3.0 and RMSE = sqrt(10.5) ≈ 3.24.", needs=[my_mae, my_rmse])

### 🧩 Problem 00.3 ⭐⭐: Beat the baseline with one tree

1. Fit a `DecisionTreeRegressor(max_depth=6, random_state=SEED)` inside a pipeline with `make_preprocessor(X_train)`.
2. Store the **test RMSE** in `tree_rmse`.
3. Store the **test RMSE of the mean-baseline** in `baseline_rmse` (hint: you can compute it without a model!).

The tree should beat the baseline by a wide margin.

<details><summary>💡 Hint 1</summary>

Follow the loop in Section 4: `make_pipeline(make_preprocessor(X_train), model)` then `.fit`, `.predict`.

</details>

<details><summary>💡 Hint 2</summary>

RMSE = `np.sqrt(np.mean((y_test - pred) ** 2))`, or `regression_report(y_test, pred)['RMSE']`.

</details>

<details><summary>💡 Hint 3</summary>

The baseline predicts `y_train.mean()` for every house; its predictions are `np.full(len(y_test), y_train.mean())`.

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [ ]:
tree_rmse = TODO       # YOUR CODE HERE
baseline_rmse = TODO   # YOUR CODE HERE

# ---- self-check (don't edit) ----
check("00.3", lambda: tree_rmse < 0.7 * baseline_rmse and baseline_rmse > 250,
      "A depth-6 tree should cut the baseline error by at least 30%.", needs=[tree_rmse, baseline_rmse])

In [14]:
scoreboard()

Scoreboard: 3 solved, 0 to fix, 0 not attempted
   ✅ 00.1
   ✅ 00.2
   ✅ 00.3


## 🏁 Recap

* Ensembles combine many models; the payoff on tabular data is large and well documented.
* We'll use three datasets (housing is the workhorse) and **one fixed train/test split** for fairness.
* **RMSE / MAE / R²** are our scorecards; baselines tell us the floor.
* A single deep tree overfits; averaging or sequentially correcting trees fixes it: *why* is Lesson 01.

### ➡️ Next up: **Lesson 01: Why Ensembles Work**
We'll simulate crowds, break the "independence" assumption on purpose, and derive the one formula
that explains bagging, random forests and everything after.

*Homework for tomorrow (10 min):* open `GLOSSARY.md`, pick three terms you couldn't define before today, and write one sentence for each in your own words.